### show me the head of a json file

In [10]:
import pandas as pd

data = pd.read_json('data/filter_all_t.json', lines=True)
## quick counts of rows

counts  = data.iloc[0][['train',"val","test"]].apply(len)
print('counts per split are:' ,counts , '\n')

#normalize each split (since each single row is a list of dicts)

train_data =pd.json_normalize(data.at[0,'train'])
val_data = pd.json_normalize(data.at[0,'val'])
test_data = pd.json_normalize(data.at[0,'test'])

print('train priview:\n', train_data.head(2),'\n')
print('val priview:\n', val_data.head(2),'\n')
print('test priview:\n', test_data.head(2),'\n')

counts per split are: train    87013
val      10860
test     11015
Name: 0, dtype: int64 

train priview:
                 business_id                user_id  rating  \
0  60567465d335d0abfb415b26  101074926318992653684       4   
1  6050fa9f5b4ccec8d5cae994  117065749986299237881       5   

                                         review_text  \
0  The tang of the tomato sauce is outstanding. A...   
1              Chicken and waffles were really good!   

                                                pics  \
0  [AF1QipM-2IRmvitARbcJr7deWfe5hyVBg_ArPMQSYvq0,...   
1     [AF1QipMpfxIZUT_aymQ3qPGO-QgGYzxbtLZGmHufAp2s]   

                                     history_reviews  
0  [[101074926318992653684_6056272797d555cc6fb0d1...  
1  [[117065749986299237881_605206f8d8c08f462b93e8...   

val priview:
                 business_id                user_id  rating  \
0  6049974fb1a0aaee3eefb0dd  112777069092124620875       5   
1  6040f95d7cd8bf1303622198  116435353904842843088       5   



In [15]:
import pandas as pd

meta_df = pd.read_json('data/image_review_all.json', lines=True)
print("Columns:", meta_df.columns.tolist())
print(meta_df.head(3))



Columns: ['business_id', 'user_id', 'rating', 'review_text', 'pics']
                business_id       user_id  rating  \
0  605730f68cd0e3d69a52284b  1.138909e+20       4   
1  605730f68cd0e3d69a52284b  1.001584e+20       5   
2  605730f68cd0e3d69a52284b  1.134952e+20       2   

                                         review_text  \
0  We came for a birthday brunch and this place i...   
1  Cool place to hang out, have drinks.  There is...   
2  This place doesn’t rock the senses when it com...   

                                                pics  
0  [{'id': 'AF1QipPrls2G30PS3tyC55KBxUrKgy3ER0AB5...  
1  [{'id': 'AF1QipPj8FEVZrdpTZmRdjoOtzQyfGYSwJ0Ub...  
2  [{'id': 'AF1QipOP5poDTRQ4XXIM11buv5x9Ae-BNXwcq...  


In [16]:
import numpy as np
import pandas as pd

def _list_len(x):
    return len(x) if isinstance(x, (list, tuple)) else 0

def summarize_split(df: pd.DataFrame, name: str):
    print(f"\n=== {name.upper()} SUMMARY ===")
    print(f"Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/1e6:.2f} MB")

    # Non-null counts & dtypes
    print("\nNon-null counts:")
    print(df.notna().sum().sort_values(ascending=False))

    print("\nDtypes:")
    print(df.dtypes)

    # Numeric summary
    num = df.select_dtypes(include="number")
    if not num.empty:
        print("\nNumeric describe():")
        print(num.describe().T)

    # Rating distribution
    if "rating" in df.columns:
        print("\nRating distribution:")
        print(df["rating"].value_counts().sort_index())
        print(f"Rating mean: {df['rating'].mean():.3f} | std: {df['rating'].std():.3f}")

    # Review text lengths & a couple samples
    if "review_text" in df.columns:
        lens = df["review_text"].fillna("").astype(str).str.len()
        print("\nreview_text length (chars):")
        print(lens.describe())
        print("\nSample review_text (3):")
        for t in df["review_text"].dropna().astype(str).head(3):
            print("-", (t[:140] + ("…" if len(t) > 140 else "")))

    # Pics / history_reviews list lengths
    if "pics" in df.columns:
        pic_len = df["pics"].apply(_list_len)
        print("\nPics per row (count):")
        print(pic_len.describe())

    if "history_reviews" in df.columns:
        hist_len = df["history_reviews"].apply(_list_len)
        print("\nhistory_reviews per row (count):")
        print(hist_len.describe())

    # Uniques & top IDs
    for col in ["business_id", "user_id"]:
        if col in df.columns:
            print(f"\nUnique {col}: {df[col].nunique()}")
            print(f"Top {col}:")
            print(df[col].value_counts().head(10))

    # Missingness %
    miss = df.isna().mean().mul(100).round(2)
    print("\nMissing % by column:")
    print(miss.sort_values(ascending=False))

# Run summaries
summarize_split(train_data, "train")
summarize_split(val_data, "val")
summarize_split(test_data, "test")



=== TRAIN SUMMARY ===
Shape: (87013, 6) | Memory: 47.15 MB

Non-null counts:
business_id        87013
user_id            87013
rating             87013
review_text        87013
pics               87013
history_reviews    87013
dtype: int64

Dtypes:
business_id        object
user_id            object
rating              int64
review_text        object
pics               object
history_reviews    object
dtype: object

Numeric describe():
          count      mean       std  min  25%  50%  75%  max
rating  87013.0  4.465252  0.833755  1.0  4.0  5.0  5.0  5.0

Rating distribution:
rating
1     1070
2     2027
3     6918
4    22333
5    54665
Name: count, dtype: int64
Rating mean: 4.465 | std: 0.834

review_text length (chars):
count    87013.000000
mean       134.239424
std        131.963632
min          0.000000
25%         54.000000
50%         93.000000
75%        168.000000
max       3456.000000
Name: review_text, dtype: float64

Sample review_text (3):
- The tang of the tomato sauce 